# Monthly screen: every step, in order

Runs the routine from `docs/guides/monthly_drop.md` for any month, one step per cell, so each step's output is visible. Run the cells top to bottom (or one at a time).

**Before you start:** put the month's files in `docs/reference/<YYYY-MM>/` (step 1 creates the folder; what to download and where it comes from is in the guide). Open this notebook with `uv run --with jupyter jupyter lab notebooks/monthly_run.ipynb`.

| Step | What it does | Needs |
|---|---|---|
| 1 | Create/inspect the month folder | |
| 2 | Merge the raw Fidelity `.xls` and Schwab `Results*.csv` bands | the raw downloads, only if you have not merged yet |
| 3 | Check every file is present and loads | |
| 4 | Build the combined per-ticker dataset | `drip_ccc.csv`, `screener_results.xlsx` |
| 5 | Refresh the market-data cache (slow, optional) | API keys in `.env` |
| 6 | Run the screen and write the reports | |
| 7 | Read the results | |

## Settings
The only cell you edit. Everything below follows from it.

In [ ]:
MONTH = ""            # "2026-11" to pick a month; "" = the newest folder under docs/reference
TOP = None            # None = the top 50 from ranking.top_n in config/criteria.toml; a number lists that many; 0 = every ranked stock, then the low-coverage ones
RUN_REFRESH = False   # step 5 takes hours the first time (about one Finnhub call per second); True to run it
REFRESH_SCOPE = "eligible"   # "eligible" (the rankable pool), "ccc" (CCC names only) or "top50"

## Setup
Finds the repo root, resolves the month, and defines `run()`, which runs a script with `uv` and shows its output as it prints.

In [ ]:
import os, re, shutil, subprocess, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "scripts").is_dir())
REFERENCE = ROOT / "docs" / "reference"
assert shutil.which("uv"), "uv is not on PATH: install it from https://docs.astral.sh/uv/"

if MONTH:
    assert re.fullmatch(r"\d{4}-\d{2}", MONTH), "MONTH must look like 2026-11"
else:
    months = sorted(p.name for p in REFERENCE.iterdir() if p.is_dir() and re.fullmatch(r"\d{4}-\d{2}", p.name))
    assert months, f"no YYYY-MM folder in {REFERENCE}: create one (step 1) and set MONTH"
    MONTH = months[-1]
FOLDER = REFERENCE / MONTH


def run(script, *args, check=True):
    """Run scripts/<script> from the repo root through uv and print its output as it arrives."""
    cmd = ["uv", "run", "--python", "3.13", f"scripts/{script}", *args]
    print("$", " ".join(cmd), "\n")
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}   # without this a piped script prints only when its buffer fills
    proc = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in proc.stdout:
        if "Pandas4Warning: The xlrd engine" not in line and not line.strip().startswith("books = [pd.read_excel"):   # the one known deprecation notice
            print(line, end="")
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"{script} failed with exit code {code}")
    return code


print(f"repo   {ROOT}\nmonth  {MONTH}\nfolder {FOLDER}")

## Step 1: the month folder
Creates `docs/reference/<MONTH>/` if it is missing and lists what is in it against the files the loaders expect (names must match exactly). Required files are `drip_ccc.csv` and `screener_results.xlsx`; the rest are optional and their criteria stay unknown when absent.

In [ ]:
FOLDER.mkdir(parents=True, exist_ok=True)
EXPECTED = {
    "drip_ccc.csv": "required: DRIP Investing (dripinvesting.org) All CCC export",
    "screener_results.xlsx": "required: Fidelity screener (saved screen div_invst2), merged in step 2",
    "morningstar_large_cap.pdf": "optional: Morningstar US Large Cap Pick List",
    "morningstar_mid_cap.pdf": "optional: Morningstar US Mid Cap Pick List",
    "argus_top30.pdf": "optional: Argus Portfolio Selector (Top 30 and Stock Screen)",
    "schwab_results.csv": "optional: Schwab screener (saved screen criteria 2), merged in step 2",
}
for name, note in EXPECTED.items():
    path = FOLDER / name
    size = f"{path.stat().st_size / 1024:8.0f} KB" if path.exists() else "   missing"
    print(f"{'ok     ' if path.exists() else '       '}{name:28} {size}   {note}")
raw_xls = sorted(FOLDER.glob("screener_results*.xls"))
raw_csv = sorted(FOLDER.glob("Results*.csv"))
print(f"\nraw Fidelity .xls files: {len(raw_xls)}   raw Schwab Results*.csv files: {len(raw_csv)}")
others = sorted(p.name for p in FOLDER.iterdir() if p.name not in EXPECTED and p not in raw_xls + raw_csv and p.name != "README.md")
if others:
    print("not read by any loader:", ", ".join(others))

## Step 2: merge the raw downloads
Fidelity and Schwab cap an export (500 and 1,000 rows), so each screen is downloaded as yield bands. These cells stitch the bands into `screener_results.xlsx` and `schwab_results.csv`. Each prints one line per band (rows and yield range): check the ranges line up with no gap. A cell is skipped when there are no raw files, so it is safe to run when you already merged. After a good merge, delete the raw files (`screener_results (N).xls`, `Results (N).csv`); a rerun needs them back.

In [ ]:
raw_xls = sorted(FOLDER.glob("screener_results*.xls"))   # scanned here so files added after step 1 are seen
if raw_xls:
    run("merge_screener.py", "--month", MONTH)
else:
    print("no raw screener_results*.xls in the folder: skipping the Fidelity merge")

In [ ]:
raw_csv = sorted(FOLDER.glob("Results*.csv"))
if raw_csv:
    run("merge_schwab.py", "--month", MONTH)
else:
    print("no raw Results*.csv in the folder: skipping the Schwab merge")

## Step 3: check the folder
One line per file: found or missing, whether it loads, and a count. Counts far from last month mean a wrong file or a changed layout; `FAILED` on a PDF means the layout changed. A missing optional file is fine. A required file that is missing or fails stops the next steps.

In [ ]:
status = run("check_month.py", "--month", MONTH, check=False)
print("\nrequired files OK" if status == 0 else "\nSTOP: fix the required file(s) above before going on")
assert status == 0

## Step 4: build the combined dataset
Joins DRIP, the Fidelity screener, Morningstar, Argus and Schwab into one row per ticker and writes `data/dataset/<MONTH>.csv`. Read the join summary: sources per ticker, names missing from the screener (look up CCC names by hand), Schwab records ignored because the name did not match, and how complete the key fields are.

In [ ]:
run("build_dataset.py", "--month", MONTH)

## Step 5: refresh the market-data cache (optional, slow)
Fills `data/cache/div_invst.sqlite` from yfinance, Finnhub, EDGAR and FRED. Only snapshots older than their max age are fetched, so a rerun is cheap and an interrupted run resumes, but the first pass takes hours. Keys (`FINNHUB_API_KEY`, `EDGAR_USER_AGENT`) come from `.env`. Criteria 1, 7, 8, 10, 11, 12 and the revision proxy read this cache and stay unknown for tickers that are not cached, so skipping it means the screen uses whatever the cache already holds. Set `RUN_REFRESH = True` in Settings to run it.

In [ ]:
if RUN_REFRESH:
    run("refresh_data.py", "--month", MONTH, "--scope", REFRESH_SCOPE)
else:
    print("skipped (RUN_REFRESH = False): the screen will use the cache as it stands")
    cache = ROOT / "data" / "cache" / "div_invst.sqlite"
    print("cache last written:", __import__("datetime").datetime.fromtimestamp(cache.stat().st_mtime).strftime("%Y-%m-%d %H:%M") if cache.exists() else "no cache yet")

## Step 6: run the screen
Runs the 15 criteria over every ticker, ranks them and writes the reports to `data/reports/<MONTH>/`: `top50.md` and `top50.csv` by default, or `ranked.md` and `ranked.csv` when `TOP` is set. The 10-year Treasury and inflation come from the cached FRED series; the S&P yield is still a manual input in `config/criteria.toml` (`[macro]`): update it monthly.

In [ ]:
REPORTS = ROOT / "data" / "reports" / MONTH
args = ["--month", MONTH] + ([] if TOP is None else ["--top", str(TOP)])
run("run_screen.py", *args)
STEM = "top50" if TOP is None else "ranked"


## Step 7: read the results
The ranked list (Y pass, n fail, ? unknown, - n/a), then how often each criterion passes.

In [ ]:
from IPython.display import Markdown, display
display(Markdown((REPORTS / f"{STEM}.md").read_text(encoding="utf-8")))

In [ ]:
import csv
rows = list(csv.DictReader(open(REPORTS / f"{STEM}.csv", encoding="utf-8")))
ranked = [r for r in rows if r["rank"]]
crit = sorted((c for c in rows[0] if re.fullmatch(r"c\d+", c)), key=lambda c: int(c[1:]))
print(f"{len(ranked)} ranked stocks listed" + (f", {len(rows) - len(ranked)} unranked (low coverage)" if len(rows) > len(ranked) else ""))
print("\ncriterion  pass  fail  unknown  n/a   (among the listed ranked stocks)")
for c in crit:
    n = [r[c] for r in ranked]
    print(f"{c:9} {n.count('pass'):5} {n.count('fail'):5} {n.count('unknown'):8} {n.count('na'):4}")

## Step 8: follow-up (by hand)
- Update `docs/reference/<MONTH>/README.md` if a file's source or format changed.
- Note anything odd in `docs/sessions/session_resume.md` (a missing optional file, a parser that needed a fix).
- Commit only when you ask: the reports and the reference folder are gitignored; the code and docs are not.